<a href="https://colab.research.google.com/github/ricardoalvarezv-cpu/Memoria-LSTM-SSI/blob/main/Ordenamiento_IN%60s_%26_OUT%60s.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**ORDENAMIENTO DE INPUTS Y OUTPUTS PARA ENTRENAMIENTO  SEQ2SEQ**

Imports + Paths base proyecto

In [ ]:
import os
import re
import glob
import datetime
import numpy as np
import pandas as pd

In [ ]:
ROOT = "/content/memoria/runs_colab"
OUT_BASE = ROOT

XLSM_PATH  = os.path.join(ROOT, "Time Series Records_MACRO.xlsm")
CASES_ROOT = os.path.join(ROOT, "Modelos x Registro")
OUT_STEP   = os.path.join(OUT_BASE, "processed_step_full_v2")

print("ROOT:", ROOT, "| exists?", os.path.exists(ROOT))
print("XLSM:", XLSM_PATH, "| exists?", os.path.exists(XLSM_PATH))
print("CASES_ROOT:", CASES_ROOT, "| exists?", os.path.exists(CASES_ROOT))
print("OUT_STEP:", OUT_STEP)

 Utilidades para leer dt correctamente

In [ ]:
def to_float_dt(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None

    if isinstance(x, (int, float, np.integer, np.floating)):
        return float(x)

    if isinstance(x, datetime.time):
        return float(x.hour * 3600 + x.minute * 60 + x.second + x.microsecond / 1e6)

    if isinstance(x, datetime.timedelta):
        return float(x.total_seconds())

    if isinstance(x, str):
        s = x.strip()
        try:
            return float(s)
        except Exception:
            parts = s.split(":")
            if len(parts) == 3:
                h, m, sec = map(float, parts)
                return h * 3600 + m * 60 + sec
            return None

    return None

Cargar registro sísmico desde Excel

In [ ]:
def load_record_by_id(xlsm_path, rec_id_1based: int):
    if rec_id_1based < 1 or rec_id_1based > 86:
        raise ValueError("rec_id debe estar entre 1 y 86")

    if rec_id_1based <= 36:
        sheet = "Registros 1-36"
        k = rec_id_1based
    elif rec_id_1based <= 72:
        sheet = "Registros 37-72"
        k = rec_id_1based - 36
    else:
        sheet = "Registros  73-"
        k = rec_id_1based - 72

    df = pd.read_excel(xlsm_path, sheet_name=sheet, header=None)
    base = 1 + 4 * (k - 1)

    dt_raw   = df.iloc[3, base + 1]
    ntps_raw = df.iloc[5, base + 1]

    dt = to_float_dt(dt_raw)
    ntps = int(ntps_raw)

    t  = df.iloc[7:7 + ntps, base + 0].to_numpy(dtype=float)
    vg = df.iloc[7:7 + ntps, base + 1].to_numpy(dtype=float)
    ag = df.iloc[7:7 + ntps, base + 2].to_numpy(dtype=float)

    if dt is None:
        dt = float(t[1] - t[0])

    return dt, ntps, t, vg, ag

Buscar carpeta del caso (ROBUSTO)

In [ ]:
def find_case_folder_by_number(cases_root, rec_id_1based: int):
    rid = rec_id_1based
    rid2 = f"{rid:02d}"
    rid3 = f"{rid:03d}"

    patterns = [
        rf"^{rid}\s*[-_]",
        rf"^{rid2}\s*[-_]",
        rf"^{rid3}\s*[-_]",
        rf"^EQ{rid3}\s*[-_]",
        rf"^EQ\s*{rid3}\s*[-_]",
        rf"Registro\s*{rid}\b",
    ]

    names = [n for n in os.listdir(cases_root)
             if os.path.isdir(os.path.join(cases_root, n))]

    for pat in patterns:
        rx = re.compile(pat, re.IGNORECASE)
        for name in names:
            if rx.search(name):
                return os.path.join(cases_root, name), name

    rx_token = re.compile(rf"(?<!\d){rid}(?!\d)")
    for name in names:
        if rx_token.search(name):
            return os.path.join(cases_root, name), name

    return None, None

In [ ]:
from pathlib import Path

def resolve_out_file(case_folder, filename):
    """
    Busca un archivo .out:
    1) directo en la carpeta del caso
    2) dentro de subcarpetas (ej. Resultados_Modelo)
    """
    case_folder = Path(case_folder)

    # intento directo
    direct = case_folder / filename
    if direct.exists():
        return str(direct)

    # búsqueda recursiva
    matches = list(case_folder.rglob(filename))

    if len(matches) == 1:
        return str(matches[0])

    elif len(matches) > 1:
        print(f"⚠️ Múltiples archivos encontrados para {filename} en {case_folder}:")
        for m in matches[:10]:
            print("   ", m)
        return str(matches[0])

    raise FileNotFoundError(f"No se encontró {filename} dentro de {case_folder}")

CELDA F — Cargar outputs OpenSees / STKO

In [ ]:
def load_case_outputs(case_folder):
    def load_out(fname):
        path = resolve_out_file(case_folder, fname)
        return np.loadtxt(path)

    # nodo de contacto
    disp_mid = load_out("Fundacion_node_disp_centro.out")

    # en algunos casos puede ser accel o acel
    try:
        vel_mid  = load_out("Fundacion_node_vel_centro.out")
    except FileNotFoundError:
        vel_mid = load_out("Fundacion_node_vel_centro.out")

    try:
        acc_mid  = load_out("Fundacion_node_accel_centro.out")
    except FileNotFoundError:
        acc_mid  = load_out("Fundacion_node_acel_centro.out")

    # aceleración base / respuesta estructural
    # si tu archivo real es otro, aquí lo ajustamos
    a_base_x = load_out("MVLEM_Accel_top.out").reshape(-1)

    # fuerzas globales del MVLEM
    Fgl = load_out("MVLEM_Fgl.out")

    Fx = Fgl[:, 0]
    Mz = Fgl[:, 2]

    return {
        "d_mid_x": disp_mid[:, 0],
        "v_mid_x": vel_mid[:, 0],
        "a_mid_x": acc_mid[:, 0],
        "theta_mid": disp_mid[:, 2],
        "a_base_x": a_base_x,
        "Fx": Fx,
        "Mz": Mz,
    }

Construir series completas por evento

In [ ]:
def build_step_full(vg, ag, out):
    NT = min(
        len(vg), len(ag),
        len(out["d_mid_x"]), len(out["v_mid_x"]), len(out["a_mid_x"]),
        len(out["a_base_x"]), len(out["Fx"]), len(out["Mz"])
    )

    X_full = np.stack([
        np.asarray(vg[:NT], dtype=np.float32),
        np.asarray(ag[:NT], dtype=np.float32),
        np.asarray(out["d_mid_x"][:NT], dtype=np.float32),
        np.asarray(out["v_mid_x"][:NT], dtype=np.float32),
        np.asarray(out["a_mid_x"][:NT], dtype=np.float32),
    ], axis=1)

    Y_full = np.stack([
        np.asarray(out["a_base_x"][:NT], dtype=np.float32),
        np.asarray(out["Fx"][:NT], dtype=np.float32),
        np.asarray(out["Mz"][:NT], dtype=np.float32),
    ], axis=1)

    theta_mid = np.asarray(out["theta_mid"][:NT], dtype=np.float32)
    return X_full, Y_full, theta_mid, NT

Loop principal de generación .npz

In [ ]:
def ensure_dir(p):
    os.makedirs(p, exist_ok=True)
    return p

ensure_dir(OUT_STEP)

# Si quieres omitir eventos, agrégalos aquí. Para correr los 86, déjalo vacío.
SKIP = set()

ok, failed = [], []

print(f"=== Generando STEP FULL NPZ => {OUT_STEP} ===")

for i in range(1, 87):
    if i in SKIP:
        continue

    tag = f"EQ{i:03d}"

    try:
        case_path, case_name = find_case_folder_by_number(CASES_ROOT, i)
        if case_path is None:
            raise FileNotFoundError("No se encontró carpeta del caso")

        dt, ntps, t, vg, ag = load_record_by_id(XLSM_PATH, i)
        out = load_case_outputs(case_path)
        X_full, Y_full, theta_mid, NT = build_step_full(vg, ag, out)

        np.savez(
            os.path.join(OUT_STEP, f"{tag}.npz"),
            X_full=X_full,
            Y_full=Y_full,
            theta_mid=theta_mid,
            dt=float(dt),
            NT=int(NT),
            record_id=tag,
            source_case_folder=case_name,
        )

        ok.append(tag)
        print(f"✅ {tag} | NT={NT} | dt={dt}")

    except Exception as e:
        failed.append((tag, str(e)))
        print(f"❌ {tag}: {e}")

print("\nRESUMEN")
print("Generados:", len(ok))
print("Fallidos:", len(failed))

if failed:
    print("Primeros fallidos:")
    for item in failed[:10]:
        print("  ", item)

Sanity check rapido

In [ ]:
files = sorted(glob.glob(os.path.join(OUT_STEP, "EQ*.npz")))
print("Total NPZ:", len(files))
assert len(files) > 0, "No se generó ningún archivo .npz"

sample = files[0]
d = np.load(sample, allow_pickle=True)

print("Archivo ejemplo:", os.path.basename(sample))
print("Keys:", list(d.files))
print("X_full:", d["X_full"].shape)
print("Y_full:", d["Y_full"].shape)
print("theta_mid:", d["theta_mid"].shape)
print("dt:", float(d["dt"]))
print("NT:", int(d["NT"]))
print("record_id:", str(d["record_id"]))

**ORDENAMIENTO DE INPUTS Y OUTPUTS PARA ENTRENAMIENTO  STEPbySTEP**

In [ ]:
# =========================
# CELDA G_STEP — Construir serie completa (STEP FULL)
# X_full: (NT, 5)  -> [vg, ag, d_mid_x, v_mid_x, a_mid_x]
# Y_full: (NT, 3)  -> [a_base_x, Fx, Mz]
# =========================
import numpy as np

def build_step_full(vg, ag, out):
    # NT común (robusto): corta a lo mínimo disponible
    NT = min(
        len(vg), len(ag),
        len(out["d_mid_x"]), len(out["v_mid_x"]), len(out["a_mid_x"]),
        len(out["a_base_x"]), len(out["Fx"]), len(out["Mz"])
    )

    X_full = np.stack([
        np.asarray(vg[:NT], dtype=np.float32),
        np.asarray(ag[:NT], dtype=np.float32),
        np.asarray(out["d_mid_x"][:NT], dtype=np.float32),
        np.asarray(out["v_mid_x"][:NT], dtype=np.float32),
        np.asarray(out["a_mid_x"][:NT], dtype=np.float32),
    ], axis=1)  # (NT,5)

    Y_full = np.stack([
        np.asarray(out["a_base_x"][:NT], dtype=np.float32),
        np.asarray(out["Fx"][:NT], dtype=np.float32),
        np.asarray(out["Mz"][:NT], dtype=np.float32),
    ], axis=1)  # (NT,3)

    theta = np.asarray(out["theta_mid"][:NT], dtype=np.float32)

    return X_full, Y_full, theta, NT


In [ ]:
# =========================
# Generación NPZ por evento (STEP FULL) [v2 inputs]
# Salida: /content/memoria/processed_step_full_v2/EQxxx.npz
# Keys: X_full, Y_full, theta_mid, dt, NT, record_id
# =========================
import os, numpy as np

SKIP = {17, 20}
OUT_STEP = os.path.join(OUT_BASE, "processed_step_full_v2")

def ensure_dir(p):
    if not os.path.isdir(p):
        os.makedirs(p)
    return p

ensure_dir(OUT_STEP)

ok, failed = [], []

print(f"=== Generando STEP FULL NPZ => {OUT_STEP} ===")

for i in range(1, 87):
    if i in SKIP:
        continue

    tag = f"EQ{i:03d}"
    try:
        case_path, _ = find_case_folder_by_number(CASES_ROOT, i)
        if case_path is None:
            raise FileNotFoundError("No se encontró carpeta")

        dt, ntps, t, vg, ag = load_record_by_id(XLSM_PATH, i)
        out = load_case_outputs(case_path)

        X_full, Y_full, theta_mid, NT = build_step_full(vg, ag, out)

        np.savez(os.path.join(OUT_STEP, f"{tag}.npz"),
                 X_full=X_full,
                 Y_full=Y_full,
                 theta_mid=theta_mid,
                 dt=float(dt),
                 NT=int(NT),
                 record_id=tag)

        ok.append(tag)

    except Exception as e:
        failed.append((tag, str(e)))
        print("❌", tag, ":", e)

print(f"\nRESUMEN: ok={len(ok)} | fallidos={len(failed)}")
if failed:
    print("Primeros fallidos:", failed[:5])


In [ ]:
# =========================
# Sanity check dataset STEP FULL
# =========================
import os, numpy as np

# ejemplo
p = os.path.join(OUT_STEP, "EQ001.npz")
assert os.path.isfile(p), f"No existe: {p}"

d = np.load(p, allow_pickle=True)
print("Archivo:", os.path.basename(p))
print("Keys:", list(d.files))

X = d["X_full"]; Y = d["Y_full"]
print("X_full:", X.shape, "| Y_full:", Y.shape)
print("dt:", float(d["dt"]), "| NT:", int(d["NT"]), "| record_id:", str(d["record_id"]))

# rangos por canal
x_names = ["vg","ag","d_mid_x","v_mid_x","a_mid_x"]
y_names = ["a_base_x","Fx","Mz"]

print("\nX_full stats:")
for j,nm in enumerate(x_names):
    v = X[:,j]
    print(f"{nm:10s} | min={v.min(): .3e} max={v.max(): .3e} mean={v.mean(): .3e} std={v.std(): .3e}")

print("\nY_full stats:")
for j,nm in enumerate(y_names):
    v = Y[:,j]
    print(f"{nm:10s} | min={v.min(): .3e} max={v.max(): .3e} mean={v.mean(): .3e} std={v.std(): .3e}")


In [ ]:
# =========================
# FREEZE: Validación + Manifest (processed_step_full_v2)
# - Valida consistencia interna de cada EQ*.npz
# - Genera MANIFEST_step_full_v2.csv con hashes + stats
# - Si DATASET_DIR es read-only, guarda el manifest en un fallback escribible
# =========================
import os, glob, hashlib
import numpy as np
import pandas as pd

DATASET_DIR = "/content/memoria/processed_step_full_v2"
assert os.path.isdir(DATASET_DIR), f"No existe {DATASET_DIR}"

files = sorted(glob.glob(os.path.join(DATASET_DIR, "EQ*.npz")))
print("Archivos detectados:", len(files))
assert len(files) > 0, "No hay archivos EQ*.npz en la carpeta"

required_keys = {"X_full","Y_full","dt","NT","record_id"}  # theta_mid opcional
rows = []
bad = []

def sha256_file(path, chunk=1<<20):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()

def is_finite_array(a):
    return np.isfinite(a).all()

for p in files:
    try:
        d = np.load(p, allow_pickle=True)
        keys = set(d.files)
        miss = required_keys - keys
        if miss:
            raise KeyError(f"Faltan keys {miss} | keys={sorted(keys)}")

        X = d["X_full"]
        Y = d["Y_full"]
        dt = float(np.array(d["dt"]).reshape(()))
        NT = int(np.array(d["NT"]).reshape(()))
        rid = str(np.array(d["record_id"]).reshape(()))

        # --- checks shape ---
        if not (isinstance(X, np.ndarray) and X.ndim == 2 and X.shape[1] == 5):
            raise ValueError(f"X_full shape raro {getattr(X,'shape',None)} (esperado (NT,5))")
        if not (isinstance(Y, np.ndarray) and Y.ndim == 2 and Y.shape[1] == 3):
            raise ValueError(f"Y_full shape raro {getattr(Y,'shape',None)} (esperado (NT,3))")

        # --- checks consistencia NT ---
        if not (X.shape[0] == Y.shape[0] == NT):
            raise ValueError(f"NT inconsistente: X{X.shape[0]} Y{Y.shape[0]} NT={NT}")

        # --- checks dt ---
        if not (dt > 0 and np.isfinite(dt)):
            raise ValueError(f"dt inválido {dt}")

        # --- checks finitud ---
        if not is_finite_array(X):
            raise ValueError("X_full contiene NaN/Inf")
        if not is_finite_array(Y):
            raise ValueError("Y_full contiene NaN/Inf")

        # stats por canal
        x_std = np.std(X, axis=0)
        y_std = np.std(Y, axis=0)

        rows.append({
            "file": os.path.basename(p),
            "record_id": rid,
            "dt": dt,
            "NT": NT,
            "has_theta_mid": ("theta_mid" in keys),
            "X_std_vg": float(x_std[0]),
            "X_std_ag": float(x_std[1]),
            "X_std_d": float(x_std[2]),
            "X_std_v": float(x_std[3]),
            "X_std_a": float(x_std[4]),
            "Y_std_a_base": float(y_std[0]),
            "Y_std_Fx": float(y_std[1]),
            "Y_std_Mz": float(y_std[2]),
            "sha256": sha256_file(p),
        })

    except Exception as e:
        bad.append((os.path.basename(p), str(e)))

df = pd.DataFrame(rows).sort_values("record_id")
print("OK:", len(df), "| BAD:", len(bad))
if bad:
    print("Primeros fallidos:", bad[:5])

# ---- elegir carpeta de salida escribible (si dataset es read-only) ----
def pick_writable_dir(preferred_dir, fallbacks):
    # prueba preferred_dir
    try:
        test = os.path.join(preferred_dir, "_write_test.tmp")
        with open(test, "w", encoding="utf-8") as f:
            f.write("ok")
        os.remove(test)
        return preferred_dir
    except Exception as e:
        print("⚠️ No puedo escribir en DATASET_DIR:", e)

    for fb in fallbacks:
        try:
            os.makedirs(fb, exist_ok=True)
            test = os.path.join(fb, "_write_test.tmp")
            with open(test, "w", encoding="utf-8") as f:
                f.write("ok")
            os.remove(test)
            return fb
        except:
            pass

    raise OSError("No encontré ninguna carpeta escribible para guardar el manifest.")

OUT_DIR = pick_writable_dir(
    DATASET_DIR,
    fallbacks=[
        "/content/memoria/runs_dataset_checks",
        "/content/memoria",
        "/content",
    ],
)

manifest_csv = os.path.join(OUT_DIR, "MANIFEST_step_full_v2.csv")
df.to_csv(manifest_csv, index=False)
print("✅ Manifest guardado:", manifest_csv)
print("📌 Nota: si OUT_DIR != DATASET_DIR, tu dataset está montado como read-only (no es error del dataset).")


In [ ]:
# =========================
# Stats globales por canal (muestra)
# =========================
import numpy as np, os, glob
import pandas as pd

DATASET_DIR = "/content/memoria/processed_step_full_v2"
files = sorted(glob.glob(os.path.join(DATASET_DIR, "EQ*.npz")))

x_names = ["vg","ag","d_mid_x","v_mid_x","a_mid_x"]
y_names = ["a_base_x","Fx","Mz"]

rows=[]
for p in files:
    d = np.load(p, allow_pickle=True)
    X = d["X_full"]; Y = d["Y_full"]
    rid = str(np.array(d["record_id"]).reshape(()))
    dt  = float(np.array(d["dt"]).reshape(()))
    NT  = int(np.array(d["NT"]).reshape(()))

    # percentiles robustos (para no depender de outliers)
    def pct(v):
        return np.percentile(v, [0, 1, 50, 99, 100])

    row={"record_id":rid,"dt":dt,"NT":NT}
    for j,nm in enumerate(x_names):
        v = X[:,j]
        p0,p1,p50,p99,p100 = pct(v)
        row[f"{nm}_p1"]=p1; row[f"{nm}_p50"]=p50; row[f"{nm}_p99"]=p99; row[f"{nm}_std"]=float(v.std())
    for j,nm in enumerate(y_names):
        v = Y[:,j]
        p0,p1,p50,p99,p100 = pct(v)
        row[f"{nm}_p1"]=p1; row[f"{nm}_p50"]=p50; row[f"{nm}_p99"]=p99; row[f"{nm}_std"]=float(v.std())
    rows.append(row)

df = pd.DataFrame(rows).sort_values("record_id")
print(df[["record_id","dt","NT","Fx_p99","Mz_p99","a_base_x_p99","d_mid_x_p99","a_mid_x_p99"]].head(10))
print("\nResumen p99 Fx (min/med/max):", df["Fx_p99"].min(), df["Fx_p99"].median(), df["Fx_p99"].max())
print("Resumen p99 Mz (min/med/max):", df["Mz_p99"].min(), df["Mz_p99"].median(), df["Mz_p99"].max())


In [ ]:
# =========================
# Cross-corr rápida (diagnóstico de desfase)
# =========================
import numpy as np, glob, os

DATASET_DIR = "/content/memoria/processed_step_full_v2"
files = sorted(glob.glob(os.path.join(DATASET_DIR, "EQ*.npz")))

def best_lag(x, y, max_lag=2000):
    # corr para lags en [-max_lag, max_lag]
    x = (x - x.mean()) / (x.std()+1e-12)
    y = (y - y.mean()) / (y.std()+1e-12)
    best = (0, -1.0)
    for lag in range(-max_lag, max_lag+1, 50):  # paso 50 para hacerlo rápido
        if lag < 0:
            xs = x[-lag:]
            ys = y[:len(xs)]
        elif lag > 0:
            xs = x[:-lag]
            ys = y[lag:]
        else:
            xs, ys = x, y
        if len(xs) < 200:
            continue
        c = float(np.mean(xs*ys))
        if abs(c) > abs(best[1]):
            best = (lag, c)
    return best

for p in files[:10]:  # revisa 10 primero
    d = np.load(p, allow_pickle=True)
    X = d["X_full"]; Y = d["Y_full"]
    rid = str(np.array(d["record_id"]).reshape(()))
    dt  = float(np.array(d["dt"]).reshape(()))

    ag = X[:,1]
    Fx = Y[:,1]
    lag, c = best_lag(ag, Fx, max_lag=2000)
    print(f"{rid} dt={dt:.5f}  best_lag(ag vs Fx)≈{lag} samples  corr≈{c:.3f}")


In [ ]:
# =========================
# FREEZE: Dataset Card (texto)
# =========================
import os
from datetime import datetime

DATASET_DIR = "/content/memoria/processed_step_full_v2"
manifest_csv = os.path.join(DATASET_DIR, "MANIFEST_step_full_v2.csv")
assert os.path.isfile(manifest_csv), "Corre Z1 primero para crear el manifest."

card = f"""DATASET CARD — processed_step_full_v2
Fecha freeze: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}

Propósito:
- Dataset continuo por evento para entrenamiento STEP-BY-STEP (TBPTT/teacher forcing opcional).
- Cada EQxxx.npz contiene series completas (sin ventanas guardadas).

Estructura por archivo EQxxx.npz:
- X_full: (NT, 5) con columnas:
  [0] vg (velocidad del registro)
  [1] ag (aceleración del registro)
  [2] d_mid_x (desplazamiento nodo contacto)
  [3] v_mid_x (velocidad nodo contacto)
  [4] a_mid_x (aceleración nodo contacto)

- Y_full: (NT, 3) con columnas:
  [0] a_base_x
  [1] Fx
  [2] Mz

- theta_mid: (NT,) opcional (guardado si existe)
- dt: escalar
- NT: escalar
- record_id: string ("EQ###")

Notas:
- Este dataset queda congelado para comparar contra baseline seq2seq.
- Manifest con hashes: {os.path.basename(manifest_csv)}
"""

path_txt = os.path.join(DATASET_DIR, "DATASET_CARD_step_full_v2.txt")
with open(path_txt, "w", encoding="utf-8") as f:
    f.write(card)

print("✅ Dataset card guardada:", path_txt)
print(card[:400], "...\n")


In [ ]:
# =========================
# Backup zip (opcional)
# =========================
import os, shutil

DATASET_DIR = "/content/memoria/processed_step_full_v2"
zip_path = "/content/memoria/processed_step_full_v2_FREEZE.zip"
if os.path.exists(zip_path):
    os.remove(zip_path)

shutil.make_archive(zip_path.replace(".zip",""), "zip", DATASET_DIR)
print("✅ ZIP creado:", zip_path)


In [ ]:
import pandas as pd

df = pd.read_csv("/content/memoria/processed_step_full_v2/MANIFEST_step_full_v2.csv")
print("dt únicos:", sorted(df["dt"].unique())[:10], "... total:", df["dt"].nunique())
print("NT min/median/max:", df["NT"].min(), int(df["NT"].median()), df["NT"].max())
print("Ejemplo HOLDOUT:", df[df["record_id"]=="EQ010"][["record_id","dt","NT"]])
